# Train YOLO26s Detector in Colab

This notebook fine-tunes `yolo26s.pt` on the license plate detector dataset.

It is a clean Colab notebook that:

- mounts Google Drive
- opens the repo from Drive
- installs detector training dependencies
- sanity-checks the current detector dataset
- writes a Colab-safe dataset YAML
- trains `yolo26s.pt` with explicit augmentation settings
- runs test-set evaluation
- saves all generated artifacts inside a fresh timestamped Google Drive folder


## Recommended Baseline

This notebook keeps the detector training settings aligned with the repo baseline while using the safer notebook structure from the newer detector experiments.

- model: `yolo26s.pt`
- epochs: `80`
- image size: `640`
- batch size: `16`
- patience: `20`

Current dataset snapshot expected by this notebook:

- train images: `329`
- val images: `86`
- test images: `85`
- class name: `plate_number`


## Augmentation Choice

This notebook uses the same mild augmentation profile as the current `yolo26n` experiment so model-size comparisons stay cleaner.

The goal is to improve robustness without distorting plate appearance too aggressively.

- keep mosaic, but not too strong: useful for a small dataset
- disable left-right flips: mirrored plate text is unrealistic
- disable up-down flips: camera views will not be upside down
- keep small color jitter and moderate scale/translation
- disable mixup and copy-paste: usually not helpful for this task
- disable random erasing: plates are small targets and easy to over-occlude


## Output Safety

This notebook is set up so every Colab run gets its own new Google Drive folder and does not reuse old output folders.

- every run gets a fresh root folder under `outputs/colab_runs/`
- training, evaluation, dataset YAML, and exported checkpoints all stay inside that one run folder
- each run name includes a timestamp
- the notebook stops if that run folder already exists
- `exist_ok=False` is passed explicitly to Ultralytics
- this notebook does not overwrite `models/detector/yolo26nbest.pt`


In [6]:
from google.colab import drive  # type: ignore[reportMissingImports]

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from datetime import datetime
from pathlib import Path

# Update this if your repo lives in a different Drive folder.
REPO_DIR = Path('/content/drive/MyDrive/plate')

DETECTOR_REQUIREMENTS_PATH = REPO_DIR / 'requirements-detector-colab.txt'
FALLBACK_REQUIREMENTS_PATH = REPO_DIR / 'requirements-colab.txt'
REQUIREMENTS_PATH = DETECTOR_REQUIREMENTS_PATH if DETECTOR_REQUIREMENTS_PATH.exists() else FALLBACK_REQUIREMENTS_PATH

MODEL_NAME = 'yolo26s.pt'
EPOCHS = 200
IMGSZ = 640
BATCH = 16
PATIENCE = 20
SEED = 42

RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_NAME = f'yolo26s_plate_{RUN_STAMP}'
RUN_ROOT = REPO_DIR / 'outputs' / 'colab_runs' / RUN_NAME
TRAIN_PROJECT = RUN_ROOT / 'training'
EVAL_PROJECT = RUN_ROOT / 'evaluation'
EXPORT_DIR = RUN_ROOT / 'exported_checkpoints'
BASE_DATA_YAML = REPO_DIR / 'configs' / 'detector_data.yaml'
DATA_YAML = RUN_ROOT / 'detector_data_colab_yolo26s.yaml'

AUGMENTATION = {
    'hsv_h': 0.01,
    'hsv_s': 0.5,
    'hsv_v': 0.3,
    'degrees': 2.0,
    'translate': 0.05,
    'scale': 0.4,
    'shear': 0.0,
    'perspective': 0.0,
    'fliplr': 0.0,
    'flipud': 0.0,
    'mosaic': 0.5,
    'close_mosaic': 10,
    'mixup': 0.0,
    'copy_paste': 0.0,
    'erasing': 0.0,
}

print('REPO_DIR =', REPO_DIR)
print('REQUIREMENTS_PATH =', REQUIREMENTS_PATH)
print('BASE_DATA_YAML =', BASE_DATA_YAML)
print('RUN_ROOT =', RUN_ROOT)
print('DATA_YAML =', DATA_YAML)
print('TRAIN_PROJECT =', TRAIN_PROJECT)
print('EVAL_PROJECT =', EVAL_PROJECT)
print('EXPORT_DIR =', EXPORT_DIR)
print('RUN_NAME =', RUN_NAME)
print('AUGMENTATION =', AUGMENTATION)


REPO_DIR = /content/drive/MyDrive/plate
REQUIREMENTS_PATH = /content/drive/MyDrive/plate/requirements-colab.txt
BASE_DATA_YAML = /content/drive/MyDrive/plate/configs/detector_data.yaml
RUN_ROOT = /content/drive/MyDrive/plate/outputs/colab_runs/yolo26s_plate_20260420_061453
DATA_YAML = /content/drive/MyDrive/plate/outputs/colab_runs/yolo26s_plate_20260420_061453/detector_data_colab_yolo26s.yaml
TRAIN_PROJECT = /content/drive/MyDrive/plate/outputs/colab_runs/yolo26s_plate_20260420_061453/training
EVAL_PROJECT = /content/drive/MyDrive/plate/outputs/colab_runs/yolo26s_plate_20260420_061453/evaluation
EXPORT_DIR = /content/drive/MyDrive/plate/outputs/colab_runs/yolo26s_plate_20260420_061453/exported_checkpoints
RUN_NAME = yolo26s_plate_20260420_061453
AUGMENTATION = {'hsv_h': 0.01, 'hsv_s': 0.5, 'hsv_v': 0.3, 'degrees': 2.0, 'translate': 0.05, 'scale': 0.4, 'shear': 0.0, 'perspective': 0.0, 'fliplr': 0.0, 'flipud': 0.0, 'mosaic': 0.5, 'close_mosaic': 10, 'mixup': 0.0, 'copy_paste': 0.0, 'er

In [8]:
assert REPO_DIR.exists(), f'Repo directory not found: {REPO_DIR}'
assert REQUIREMENTS_PATH.exists(), f'Requirements file not found: {REQUIREMENTS_PATH}'
assert BASE_DATA_YAML.exists(), f'Detector dataset YAML not found: {BASE_DATA_YAML}'
assert not RUN_ROOT.exists(), (
    f'Run folder already exists: {RUN_ROOT}. Rerun the config cell to generate a fresh timestamped folder.'
)

print('Repo ready:', REPO_DIR)


Repo ready: /content/drive/MyDrive/plate


In [9]:
import subprocess
import sys

print('Installing from', REQUIREMENTS_PATH)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REQUIREMENTS_PATH)], check=True)


Installing from /content/drive/MyDrive/plate/requirements-colab.txt


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-r', '/content/drive/MyDrive/plate/requirements-colab.txt'], returncode=0)

In [10]:
from pathlib import Path
import yaml

EXPECTED_COUNTS = {'train': 329, 'val': 86, 'test': 85}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

base_config = yaml.safe_load(BASE_DATA_YAML.read_text(encoding='utf-8'))
if not isinstance(base_config, dict):
    print(f'warning: unexpected detector YAML structure in {BASE_DATA_YAML}: {type(base_config)!r}')
    base_config = {}

names_field = base_config.get('names')
if isinstance(names_field, dict):
    class_names = [name for _, name in sorted(names_field.items(), key=lambda item: int(item[0]))]
elif isinstance(names_field, list):
    class_names = list(names_field)
elif base_config.get('nc') == 1:
    class_names = ['plate_number']
    print(f'warning: {BASE_DATA_YAML} has no usable names field; falling back to nc=1 -> plate_number')
else:
    class_names = ['plate_number']
    print(f'warning: {BASE_DATA_YAML} has no usable names field; falling back to repo default class name plate_number')

assert class_names == ['plate_number'], f'Unexpected class names: {class_names}'

path_value = base_config.get('path')
if isinstance(path_value, str) and path_value.strip():
    data_root = (BASE_DATA_YAML.parent / path_value).resolve()
else:
    data_root = (REPO_DIR / 'data').resolve()
    print(f'warning: {BASE_DATA_YAML} has no usable path field; falling back to {data_root}')
print('Resolved data root:', data_root)

for split, expected_count in EXPECTED_COUNTS.items():
    image_dir = data_root / 'images' / split
    label_dir = data_root / 'labels' / split
    assert image_dir.exists(), f'Missing image directory: {image_dir}'
    assert label_dir.exists(), f'Missing label directory: {label_dir}'

    image_files = sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
    label_files = sorted(label_dir.glob('*.txt'))

    print(f'{split}: images={len(image_files)} labels={len(label_files)} expected_images={expected_count}')
    assert len(image_files) == len(label_files), (
        f'{split} image/label count mismatch: images={len(image_files)} labels={len(label_files)}'
    )
    if len(image_files) != expected_count:
        print(f'  warning: current image count differs from the checked-in snapshot ({expected_count})')

print('Detector dataset sanity check passed.')


Resolved data root: /content/drive/MyDrive/plate/data
train: images=329 labels=329 expected_images=329
val: images=86 labels=86 expected_images=86
test: images=85 labels=85 expected_images=85
Detector dataset sanity check passed.


In [11]:
import yaml

RUN_ROOT.mkdir(parents=True, exist_ok=False)
TRAIN_PROJECT.mkdir(parents=True, exist_ok=False)
EVAL_PROJECT.mkdir(parents=True, exist_ok=False)
EXPORT_DIR.mkdir(parents=True, exist_ok=False)

colab_dataset = {
    'path': str(REPO_DIR / 'data'),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': ['plate_number'],
}

DATA_YAML.write_text(yaml.safe_dump(colab_dataset, sort_keys=False), encoding='utf-8')
print('Wrote dataset YAML to:', DATA_YAML)
print(DATA_YAML.read_text(encoding='utf-8'))


Wrote dataset YAML to: /content/drive/MyDrive/plate/outputs/colab_runs/yolo26s_plate_20260420_061453/detector_data_colab_yolo26s.yaml
path: /content/drive/MyDrive/plate/data
train: images/train
val: images/val
test: images/test
names:
- plate_number



In [ ]:
from pathlib import Path
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(TRAIN_PROJECT),
    name='train',
    patience=PATIENCE,
    seed=SEED,
    exist_ok=False,
    **AUGMENTATION,
)

RUN_DIR = Path(results.save_dir)
print('Training run directory:', RUN_DIR)
results


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/plate/outputs/colab_runs/yolo26s_plate_20260420_061453/detector_data_colab_yolo26s.yaml, degrees=2.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchsc

: 0% ──────────── 0/21  10.3s


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
from ultralytics import YOLO

run_dir = Path(results.save_dir) if 'results' in globals() else TRAIN_PROJECT / 'train'
assert run_dir.exists(), f'Training run directory not found: {run_dir}'

best_pt = run_dir / 'weights' / 'best.pt'
assert best_pt.exists(), f'best.pt not found at {best_pt}'

best_model = YOLO(str(best_pt))
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    project=str(EVAL_PROJECT),
    name='test_eval',
    exist_ok=False,
)

test_metrics.results_dict


In [ ]:
from pathlib import Path
import shutil

run_dir = Path(results.save_dir) if 'results' in globals() else TRAIN_PROJECT / 'train'
assert run_dir.exists(), f'Training run directory not found: {run_dir}'

best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'
assert best_pt.exists(), f'best.pt not found at {best_pt}'

target_dir = EXPORT_DIR

target_best = target_dir / 'best.pt'
target_last = target_dir / 'last.pt'

assert not target_best.exists(), f'Target best checkpoint already exists: {target_best}'
shutil.copy2(best_pt, target_best)
print('Copied best checkpoint to:', target_best)

if last_pt.exists():
    assert not target_last.exists(), f'Target last checkpoint already exists: {target_last}'
    shutil.copy2(last_pt, target_last)
    print('Copied last checkpoint to:', target_last)

print('Run output root:', RUN_ROOT)
print('Stable detector alias left untouched:', REPO_DIR / 'models' / 'detector' / 'yolo26nbest.pt')


In [ ]:
from pathlib import Path

run_dir = Path(results.save_dir) if 'results' in globals() else TRAIN_PROJECT / 'train'
assert run_dir.exists(), f'Training run directory not found: {run_dir}'

test_eval_dir = EVAL_PROJECT / 'test_eval'
candidate_dir = EXPORT_DIR
print('Run root directory:', RUN_ROOT)
print('Run directory:', run_dir)
print('Test evaluation directory:', test_eval_dir)
print('Exported checkpoint directory:', candidate_dir)
print('Weights directory contents:')
for path in sorted((run_dir / 'weights').glob('*')):
    print('-', path)


## Optional Promotion Step

This notebook intentionally does not replace `models/detector/yolo26nbest.pt`.

After you compare the new `yolo26s` run against your existing candidate models, you can manually promote the chosen checkpoint later from the run folder's `exported_checkpoints/` directory if it is actually better.


## After Training

When this run finishes:

1. review validation metrics and training plots
2. review test-set metrics from the evaluation cell
3. compare the new `yolo26s` result against your `yolo26n` and active detector baselines
4. keep the preferred checkpoint from `exported_checkpoints/` only after manual review
5. continue with OCR and end-to-end evaluation using the rest of the repo workflow
